# Annotate spikes on cleaned .edf files

## Import recordings

Load packages

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import scipy
from scipy import signal
import os
from scipy.stats import zscore
from ephyviewer import mkQApp, MainViewer, TraceViewer, CsvEpochSource, InMemoryAnalogSignalSource, EpochEncoder,TimeFreqViewer,AnalogSignalSourceWithScatter, EpochEncoder_ABC
from matplotlib import cm
from matplotlib.colors import to_hex
import re
import mne
import pandas as pd
import numpy as np
import csv
from collections import defaultdict
import ast
import matplotlib
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import string
from IPython.display import display
from ipyfilechooser import FileChooser
import ipywidgets as widgets
%matplotlib widget

Choose .edf file

In [15]:
dpath = "//10.69.168.1/crnldata/forgetting/Aurelie/EpiKid/CleanedData/"
fc1 = FileChooser(dpath,select_default=True, show_only_dirs = False, title = "<b>Choose a .edf file </b>", layout=widgets.Layout(width='100%'))
fc1.filter_pattern = '*.edf'
display(fc1)
def update_my_folder(chooser):
    global dpath
    dpath = chooser.selected
    %store dpath
    return 
fc1.register_callback(update_my_folder)

FileChooser(path='\\10.69.168.1\crnldata\forgetting\Aurelie\EpiKid\CleanedData', filename='', title='<b>Choose…

Load cleaned edf file

In [17]:
folder_base = Path(dpath) 
data = mne.io.read_raw_edf(dpath, preload=False)
raw_data = data.get_data() 

info = data.info
channels = data.ch_names
timestamps = data.times
duration = data.duration
samplerate = data.info.get('sfreq')  # in Hz
data.info.get('subject_info')
lowpass=data.info.get('lowpass')
highpass=data.info.get('highpass')

combined = raw_data.T
print(f'{round(np.shape(raw_data)[1]/samplerate/60/60,3)}h recording if sampled at {samplerate}Hz ({duration} seconds)')
print(f'{np.shape(raw_data)[0]} channels found: {channels}')

Extracting EDF parameters from \\10.69.168.1\crnldata\forgetting\Aurelie\EpiKid\CleanedData\AGAL06-E_06032017_repaired_montagePos.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
10.601h recording if sampled at 256.0Hz (38164.0 seconds)
20 channels found: ['Fp2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'Fp2-F8', 'F8-T4', 'T4-TB2', 'T6-O2', 'Fp1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'Fp1-F7', 'F7-T3', 'T3-TB1', 'T5-O1', 'Fz-Cz', 'Cz-Pz', 'EMG1+', 'EMG2+']


Zscore & filter signals

In [18]:
montage_eeg=[]
montage_name=[]
for i, ch in enumerate(channels):
    if ch.count('-')==0:
        ch1 = ch
        eeg=combined[:,i]
        montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        montage_name.append(ch)
    else:        
        ch1 = ch
        eeg=combined[:,i]
        nyq = samplerate / 2
        f_lowcut = 0.5
        f_hicut = 35.0 #70
        Wn = [f_lowcut/nyq, f_hicut/nyq]
        sos = signal.butter(6, Wn, btype='band', output='sos')
        eeg = signal.sosfiltfilt(sos, eeg)
        montage_eeg = zscore(eeg[:,np.newaxis]) if len(montage_eeg) == 0 else np.append(montage_eeg, zscore(eeg[:,np.newaxis]), axis=1)
        montage_name.append(ch)
montage_name_eeg=montage_name[:-2]

## Annotate spikes as epoch on montage

In [23]:
labels = ["WAKE", "Stage N1", "Stage N2", "Stage N3", "REM", "Unstaged"]

epoch_dur = 30 # define epoch duration in sec
spike_dur = .01 # define epoch duration in sec
winlen = 30 # default window length in sec

file = ('_').join(folder_base.name.split("_")[:2])
SleepScoring_filename = f'{Path(folder_base).parent}/EphyViewer_Scoring_{file}.csv'
#SleepScoring_filename = f'{dpath}/Sleep_Scoring_{len(labels)}Stages_{epoch_dur}sEpoch.csv'
source_epoch = CsvEpochSource(SleepScoring_filename, labels)

Spikes_filename = f'{Path(folder_base).parent}/EphyViewer_Spikes_{file}.csv'
#SleepScoring_filename = f'{dpath}/Sleep_Scoring_{len(labels)}Stages_{epoch_dur}sEpoch.csv'
source_epoch2 = CsvEpochSource(Spikes_filename, montage_name)


#you must first create a main Qt application (for event loop)
app = mkQApp()

t_start = 0.

#Create the main window that can contain several viewers
win = MainViewer(debug=False, show_auto_scale=True)

#create a viewer for signal
ch_names = [f"{s} ({string.ascii_uppercase[i]})" for i, s in enumerate(montage_name)]
source =InMemoryAnalogSignalSource(montage_eeg, samplerate, t_start, channel_names=ch_names)
view1 = TraceViewer(source=source, name='Signal')

view1.params['xsize']= winlen
view1.params['display_labels'] = True
view1.params['background_color'] = "#FFFFFFFF"
view1.params['label_fill_color'] = "#FFFFFFFF"
view1.params['vline_color'] = "#0000006A"

view1.params['scale_mode'] = 'same_for_all'
colormap = ["#000000"]* 50
for idx, ch in enumerate(montage_name): 
    view1.by_channel_params[f'ch{idx}', 'color'] = colormap[idx] #FF0000 red, #00FF00 green, and #0000FF blue
view1.auto_scale()


#create a viewer for the encoder itself
view2 = EpochEncoder(source=source_epoch, name='Sleep Scoring')

view2.params['xsize'] = winlen
view2.params['new_epoch_step'] = epoch_dur

view2.by_label_params['label0', 'color'] = "#09ff00" 
view2.by_label_params['label1', 'color'] = "#00f7ff"
view2.by_label_params['label2', 'color'] = "#fffb00"
view2.by_label_params['label3', 'color'] = "#ff00d4"
view2.by_label_params['label4', 'color'] = "#ff0000"
view2.by_label_params['label5', 'color'] = "#8b8b8b"

view2.params['background_color'] = "#FFFFFF00"
view2.params['label_fill_color'] = "#FFFFFF00"
view2.params['vline_color'] = "#0000006A"

view2.params['view_mode'] = 'flat'
view2.controls.hide()



#create a viewer for the encoder itself
view4 = EpochEncoder_ABC(source=source_epoch2, name='Spikes')
view4.params['xsize'] = winlen
view4.params['new_epoch_step'] = spike_dur
view4.params['background_color'] = "#FFFFFFFF"
view4.params['label_fill_color'] = "#FFFFFFFF"
view4.params['vline_color'] = "#0000006A"

colormap = ["#FF0000"]* 50
for idx, ch in enumerate(montage_name): 
    view4.by_label_params[f'label{idx}', 'color'] = colormap[idx]
    
view4.params['exclusive_mode'] = False # Allow overlap
view4.controls.hide()



win.add_view(view1)
win.add_view(view2)
win.add_view(view4, orientation='horizontal')

#win.set_overlay_opacity('Spikes', opacity=.5)
#win.set_overlay_opacity('Signal', opacity=0.5)

win.navigation_toolbar.spinbox_xsize.setValue(winlen)
win.show()

app.exec()

0